<a href="https://colab.research.google.com/github/Cha5machou/AML1/blob/master/Hina_Mod_AICoverGen_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# AICoverGen WebUI (Modded by [Hina](https://discordlookup.com/user/444684887363026974))

Simply click `Runtime` in the top navigation bar and `Run all`. Wait for the output of the final cell to show the public gradio url and click on it.

In [ ]:
#@title Clone repository
from IPython.display import clear_output, Javascript
import codecs
import threading
import time
import os
# cloneing=codecs.decode('uggcf://tvguho.pbz/uvanoy/NVPbireTra-Pbyno.tvg','rot_13')

#=======================Auto Edit======================

#@markdown ---
#@markdown Switch between ```-1 0 1``` or ```-12 0 12``` pitch change control

#@markdown This can only be changed once, you need to restart the whole thing if you wanna change it again
Pitch_Change="12" #@param ['1','12']

#@markdown Enable if you want to install the Program to your Drive
if Pitch_Change=="1":
  cloneing=codecs.decode('uggcf://tvguho.pbz/FbpvnyylVarcgJrro/NVPbireTra.tvg','rot_13')
else:
  cloneing=codecs.decode('uggcf://tvguho.pbz/nequn27/NVPbireTra-Zbq.tvg','rot_13')
#=====================Auto Edit End================
Install_To_Drive=True #@param {type:"boolean"}


#====================Use Drive============
if Install_To_Drive == True:
    from google.colab import drive
    drive.mount('/content/drive')
    repo_path = '/content/drive/MyDrive/Hina_RVC'
else:
    repo_path = '/content/Hina_RVC'

if os.path.isdir(repo_path):
    print(f"Repository already exists at {repo_path}, pulling the latest changes.")
    %cd $repo_path
    !git pull
else:
    print(f"Cloning repository to {repo_path}")
    !git clone $cloneing Hina_RVC
    if Install_To_Drive == True:
        !mv Hina_RVC /content/drive/MyDrive/
        %cd /content/drive/MyDrive/Hina_RVC
    else:
        %cd Hina_RVC

!rm -rf sample_data


def update_timer_and_print():
    global timer
    while True:
        hours, remainder = divmod(timer, 3600)
        minutes, seconds = divmod(remainder, 60)
        timer_str = f'{hours:02}:{minutes:02}:{seconds:02}'
        print(f'\rTimer: {timer_str}', end='', flush=True)  # Print without a newline
        time.sleep(1)
        timer += 1
timer = 0
threading.Thread(target=update_timer_and_print, daemon=True).start()



clear_output()
print("Done Cloning Repository")

In [ ]:
#@title Install requirements
!sed -i '/torch==/d' requirements.txt
!sed -i '/torchaudio==/d' requirements.txt
!sed -i '/numpy==/d' requirements.txt
!sed -i '/librosa==/d' requirements.txt
!sed -i '/Requests==/d' requirements.txt
!sed -i '/scipy==/d' requirements.txt
!sed -i '/soundfile==/d' requirements.txt
!sed -i '/tqdm==/d' requirements.txt
!sed -i '/onnxruntime_gpu/d' requirements.txt

!pip install pip==23.3.1
!pip install -r requirements.txt
!pip install -q ort-nightly-gpu --index-url=https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/ort-cuda-12-nightly/pypi/simple/
!pip install fastapi==0.112.2
clear_output()
print("Finished Installing Requirements")
!sudo apt update
clear_output()
print("Finished Updating")
!sudo apt install sox
clear_output()
print("Finsihed running this cell, proceed to the next cell")

In [ ]:
!pip install git+https://github.com/liyaodev/fairseq.git

In [ ]:
#@title Download MDXNet Vocal Separation and Hubert Base Models
models=codecs.decode('fep/qbjaybnq_zbqryf.cl','rot_13')
!python $models
clear_output()
print("Finished Downloading Voice Separation Model and Hubert Base Model")

In [ ]:
#@title Run WebUI
runpice=codecs.decode('fep/jrohv.cl','rot_13')
!python $runpice --share

[![](https://i.pinimg.com/474x/de/72/9e/de729ecfa41b69901c42c82fff752414.jpg)](https://discordlookup.com/user/444684887363026974)

In [ ]:
import os
BASE_DIR = "/content/drive/MyDrive/Hina_RVC"
mdxnet_models_dir = os.path.join(BASE_DIR, 'mdxnet_models')
rvc_models_dir = os.path.join(BASE_DIR, 'rvc_models')
output_dir = os.path.join(BASE_DIR, 'song_output')

song_input = "/content/drive/MyDrive/Halim/infer/original.mp3"
voice_model = "Ahmed"
pitch_change = -12
keep_files=True
index_rate = 0.5
filter_radius = 3
rms_mix_rate = 0.25
f0_method = 'rmvpe'
crepe_hop_length = 128
protect = 0.33
main_vol = 0,
backup_vol = 0
inst_vol = 0,
pitch_change_all = 0
reverb_size = 0.15
reverb_wetness = 0.2
reverb_dryness = 0.8
reverb_damping = 0.7
output_format='mp3'

In [ ]:
%cd /content/drive/MyDrive/Hina_RVC/src/

In [ ]:
import json
import hashlib
import librosa
import shlex
import subprocess
import numpy as np
import gc
from mdx import run_mdx, run_roformer
from rvc import Config, load_hubert, get_vc, rvc_infer
def get_hash(filepath):
    with open(filepath, 'rb') as f:
        file_hash = hashlib.blake2b()
        while chunk := f.read(8192):
            file_hash.update(chunk)

    return file_hash.hexdigest()[:11]
def get_audio_paths(song_dir):
    orig_song_path = None
    instrumentals_path = None
    main_vocals_dereverb_path = None
    backup_vocals_path = None

    for file in os.listdir(song_dir):
        if file.endswith('_Instrumental.wav'):
            instrumentals_path = os.path.join(song_dir, file)
            orig_song_path = instrumentals_path.replace('_Instrumental', '')

        elif file.endswith('_Vocals_Main_DeReverb.wav'):
            main_vocals_dereverb_path = os.path.join(song_dir, file)

        elif file.endswith('_Vocals_Backup.wav'):
            backup_vocals_path = os.path.join(song_dir, file)

    return orig_song_path, instrumentals_path, main_vocals_dereverb_path, backup_vocals_path

def display_progress(message, percent, is_webui, progress=None):
      print(message)
def convert_to_stereo(audio_path):
  wave, sr = librosa.load(audio_path, mono=False, sr=44100)

  # check if mono
  if type(wave[0]) != np.ndarray:
      stereo_path = f'{os.path.splitext(audio_path)[0]}_stereo.wav'
      command = shlex.split(f'ffmpeg -y -loglevel error -i "{audio_path}" -ac 2 -f wav "{stereo_path}"')
      subprocess.run(command)
      return stereo_path
  else:
      return audio_path
def preprocess_song(song_input, mdx_model_params, song_id, is_webui, input_type, progress=None):
    orig_song_path = song_input
    keep_orig = True


    song_output_dir = os.path.join(output_dir, song_id)
    orig_song_path = convert_to_stereo(orig_song_path)

    display_progress('[~] Separating Vocals from Instrumental...', 0.1, is_webui, progress)
    vocals_path, instrumentals_path = run_roformer(mdx_model_params, song_output_dir, 'model_bs_roformer_ep_317_sdr_12.9755.ckpt', orig_song_path, denoise=True, keep_orig=keep_orig)

    display_progress('[~] Separating Main Vocals from Backup Vocals...', 0.2, is_webui, progress)
    backup_vocals_path, main_vocals_path = run_mdx(mdx_model_params, song_output_dir, os.path.join(mdxnet_models_dir, 'UVR_MDXNET_KARA_2.onnx'), vocals_path, suffix='Backup', invert_suffix='Main', denoise=True)

    display_progress('[~] Applying DeReverb to Vocals...', 0.3, is_webui, progress)
    _, main_vocals_dereverb_path = run_mdx(mdx_model_params, song_output_dir, os.path.join(mdxnet_models_dir, 'Reverb_HQ_By_FoxJoy.onnx'), main_vocals_path, invert_suffix='DeReverb', exclude_main=True, denoise=True)

    return orig_song_path, vocals_path, instrumentals_path, main_vocals_path, backup_vocals_path, main_vocals_dereverb_path

def get_rvc_model(voice_model, is_webui):
    rvc_model_filename, rvc_index_filename = None, None
    model_dir = os.path.join(rvc_models_dir, voice_model)
    for file in os.listdir(model_dir):
        ext = os.path.splitext(file)[1]
        if ext == '.pth':
            rvc_model_filename = file
        if ext == '.index':
            rvc_index_filename = file

    if rvc_model_filename is None:
        error_msg = f'No model file exists in {model_dir}.'
        raise Exception(error_msg)

    return os.path.join(model_dir, rvc_model_filename), os.path.join(model_dir, rvc_index_filename) if rvc_index_filename else ''

def voice_change(voice_model, vocals_path, output_path, pitch_change, f0_method, index_rate, filter_radius, rms_mix_rate, protect, crepe_hop_length, is_webui):
    rvc_model_path, rvc_index_path = get_rvc_model(voice_model, is_webui)
    device = 'cuda:0'
    config = Config(device, True)
    hubert_model = load_hubert(device, config.is_half, os.path.join(rvc_models_dir, 'hubert_base.pt'))
    cpt, version, net_g, tgt_sr, vc = get_vc(device, config.is_half, config, rvc_model_path)

    # convert main vocals
    rvc_infer(rvc_index_path, index_rate, vocals_path, output_path, pitch_change, f0_method, cpt, version, net_g, filter_radius, tgt_sr, rms_mix_rate, protect, crepe_hop_length, vc, hubert_model)
    del hubert_model, cpt
    gc.collect()

In [ ]:
is_webui=False
input_type = 'local'
with open(os.path.join(mdxnet_models_dir, 'model_data.json')) as infile:
    mdx_model_params = json.load(infile)
song_input = song_input.strip('\"')
if os.path.exists(song_input):
    song_id = get_hash(song_input)
song_dir = os.path.join(output_dir, song_id)

if not os.path.exists(song_dir):
    os.makedirs(song_dir)
    orig_song_path, vocals_path, instrumentals_path, main_vocals_path, backup_vocals_path, main_vocals_dereverb_path = preprocess_song(song_input, mdx_model_params, song_id, is_webui, input_type)

else:
    vocals_path, main_vocals_path = None, None
    paths = get_audio_paths(song_dir)

    # if any of the audio files aren't available or keep intermediate files, rerun preprocess
    if any(path is None for path in paths) or keep_files:
        orig_song_path, vocals_path, instrumentals_path, main_vocals_path, backup_vocals_path, main_vocals_dereverb_path = preprocess_song(song_input, mdx_model_params, song_id, is_webui, input_type)
    else:
        orig_song_path, instrumentals_path, main_vocals_dereverb_path, backup_vocals_path = paths

pitch_change = pitch_change + pitch_change_all
ai_vocals_path = os.path.join(song_dir, f'{os.path.splitext(os.path.basename(orig_song_path))[0]}_lead_{voice_model}_p{pitch_change}_i{index_rate}_fr{filter_radius}_rms{rms_mix_rate}_pro{protect}_{f0_method}{"" if f0_method != "mangio-crepe" else f"_{crepe_hop_length}"}.wav')
ai_backing_path = os.path.join(song_dir, f'{os.path.splitext(os.path.basename(orig_song_path))[0]}_backing_{voice_model}_p{pitch_change}_i{index_rate}_fr{filter_radius}_rms{rms_mix_rate}_pro{protect}_{f0_method}{"" if f0_method != "mangio-crepe" else f"_{crepe_hop_length}"}.wav')

ai_cover_path = os.path.join(song_dir, f'{os.path.splitext(os.path.basename(orig_song_path))[0]} ({voice_model} Ver).{output_format}')
ai_cover_backing_path = os.path.join(song_dir, f'{os.path.splitext(os.path.basename(orig_song_path))[0]} ({voice_model} Ver With Backing).{output_format}')

In [ ]:
if not os.path.exists(ai_vocals_path):
  display_progress('[~] Converting lead voice using RVC...', 0.5, is_webui, progress)
  voice_change(voice_model, main_vocals_dereverb_path, ai_vocals_path, pitch_change, f0_method, index_rate, filter_radius, rms_mix_rate, protect, crepe_hop_length, is_webui)

  display_progress('[~] Converting backing voice using RVC...', 0.65, is_webui, progress)
  voice_change(voice_model, backup_vocals_path, ai_backing_path, pitch_change, f0_method, index_rate, filter_radius, rms_mix_rate, protect, crepe_hop_length, is_webui)

display_progress('[~] Applying audio effects to Vocals...', 0.8, is_webui, progress)
ai_vocals_mixed_path = add_audio_effects(ai_vocals_path, reverb_rm_size, reverb_wet, reverb_dry, reverb_damping)
ai_backing_mixed_path = add_audio_effects(ai_backing_path, reverb_rm_size, reverb_wet, reverb_dry, reverb_damping)

if pitch_change_all != 0:
  display_progress('[~] Applying overall pitch change', 0.85, is_webui, progress)
  instrumentals_path = pitch_shift(instrumentals_path, pitch_change_all)
  backup_vocals_path = pitch_shift(backup_vocals_path, pitch_change_all)

display_progress('[~] Combining AI Vocals and Instrumentals...', 0.9, is_webui, progress)
combine_audio([ai_vocals_mixed_path, backup_vocals_path, instrumentals_path], ai_cover_path, main_gain, backup_gain, inst_gain, output_format)
combine_audio([ai_vocals_mixed_path, ai_backing_mixed_path, instrumentals_path], ai_cover_backing_path, main_gain, backup_gain, inst_gain, output_format)

if not keep_files:
  display_progress('[~] Removing intermediate audio files...', 0.95, is_webui, progress)
  intermediate_files = [vocals_path, main_vocals_path, ai_vocals_mixed_path, ai_backing_mixed_path]
  if pitch_change_all != 0:
      intermediate_files += [instrumentals_path, backup_vocals_path]
  for file in intermediate_files:
      if file and os.path.exists(file):
          os.remove(file)